# 🤖 SYNTHETIC INSTRUCTION DATASET GENERATION FOR LLM FINE-TUNING

**Fase 2: Menghubungkan Prediksi Tabular ML + SHAP XAI Menjadi Dataset Instruksi Fine-Tuning (Alpaca / JSONL Format)**

---

### 🎯 Tujuan Pembelajaran & Output Notebook Ini
Notebook ini dirancang sebagai jembatan (*bridge*) antara **Predictive ML (XGBoost + SHAP)** dengan **Generative AI (Fine-Tuning Qwen 2.5-7B)**.

Di sini Anda akan menyusun pipeline otomatis yang mengonversi profil pemohon pinjaman beserta perhitungan risiko matematisnya menjadi ribuan pasangan data instruksi berformat **Alpaca Prompt** (`instruction`, `input`, `output`).

```
+------------------------------------------------------------------------------------------------+
|                                    ALUR GENERASI DATASET                                       |
|                                                                                                |
|  [1. Profil Nasabah]  +  [2. Prediksi XGBoost]  +  [3. Kontribusi SHAP]                        |
|  (Age, Income, Loan)     (Probability of Default)  (Top Risk Drivers & Mitigating Factors)     |
|                                       │                                                        |
|                                       ▼                                                        |
|                       [Prompt & Logic Synthesis Engine]                                        |
|                                       │                                                        |
|                                       ▼                                                        |
|                   [Alpaca Format Instruction Pair (JSONL)]                                     |
|   - Instruction: Panduan Underwriter AI                                                       |
|   - Input: Ringkasan profil & sinyal risiko matematis                                         |
|   - Output: Structured Underwriting Memo (JSON Standard)                                      |
+------------------------------------------------------------------------------------------------+
```

---

### 📋 Daftar Isi / Table of Contents
1. [Setup & Import Libraries](#1.-Setup-&-Import-Libraries)
2. [Load Model XGBoost, Metadata & Dataset Preprocessed](#2.-Load-Model-XGBoost,-Metadata-&-Dataset-Preprocessed)
3. [Inisialisasi SHAP TreeExplainer & Engine Ekstraksi Fitur](#3.-Inisialisasi-SHAP-TreeExplainer-&-Engine-Ekstraksi-Fitur)
4. [Desain Template Prompt & Rule-Based Underwriting Logic](#4.-Desain-Template-Prompt-&-Rule-Based-Underwriting-Logic)
5. [Sintesis & Sampling Dataset (Balanced 1,000+ Records)](#5.-Sintesis-&-Sampling-Dataset-(Balanced-1,000+-Records))
6. [Validasi Kualitas Data & JSON Schema Verification](#6.-Validasi-Kualitas-Data-&-JSON-Schema-Verification)
7. [Export ke Format JSONL (`data/credit_finetune_dataset.jsonl`)](#7.-Export-ke-Format-JSONL)

## 1. Setup & Import Libraries

**Penjelasan Konsep:**
Kita membutuhkan pustaka manipulasi data (`pandas`, `numpy`), serialisasi model (`joblib`), manipulasi string/format JSON (`json`, `re`), pelacak progress bar (`tqdm`), serta `shap` untuk mengekstrak kontribusi lokal setiap record nasabah.

In [1]:
import os
import json
import random
import joblib
import numpy as np
import pandas as pd
import shap
from tqdm import tqdm

# Set random seed untuk memastikan reproduksibilitas dataset
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Setup selesai! Pustaka berhasil dimuat.")

Setup selesai! Pustaka berhasil dimuat.


## 2. Load Model XGBoost, Metadata & Dataset Preprocessed

**Penjelasan Konsep:**
Pada langkah ini kita memuat:
1. `models/credit_xgboost_model.pkl`: Model klasifikasi yang telah dilatih pada notebook `02`.
2. `models/model_metadata.json`: Informasi daftar urutan nama fitur dan metrik.
3. `data/preprocessed_credit_risk.csv`: Dataset fitur numerik siap inferensi.
4. `data/cleaned_dataset.csv` (opsional): Dataset dalam format teks kategorikal asli (misal: status rumah `'RENT'`, tujuan pinjaman `'EDUCATION'`) untuk mempermudah penyusunan teks prompt yang terbaca manusia (*human-readable*).

In [2]:
# Path direktori data dan model
model_path = '../models/credit_xgboost_model.pkl' if os.path.exists('../models/credit_xgboost_model.pkl') else 'models/credit_xgboost_model.pkl'
metadata_path = '../models/model_metadata.json' if os.path.exists('../models/model_metadata.json') else 'models/model_metadata.json'
preprocessed_data_path = '../data/preprocessed_credit_risk.csv' if os.path.exists('../data/preprocessed_credit_risk.csv') else 'data/preprocessed_credit_risk.csv'
cleaned_data_path = '../data/cleaned_dataset.csv' if os.path.exists('../data/cleaned_dataset.csv') else 'data/cleaned_dataset.csv'

# TODO 1: Muat model XGBoost menggunakan joblib.load()
model = joblib.load(model_path)

# TODO 2: Muat metadata JSON
with open(metadata_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)
feature_names = metadata['features']

# TODO 3: Muat dataset preprocessed dan cleaned
df_prep = pd.read_csv(preprocessed_data_path)
df_clean = pd.read_csv(cleaned_data_path)

print(f"Model dimuat dari: {model_path}")
print(f"Jumlah fitur model: {len(feature_names)}")
print(f"Dimensi data preprocessed: {df_prep.shape}")
print(f"Dimensi data cleaned: {df_clean.shape}")

Model dimuat dari: ../models/credit_xgboost_model.pkl
Jumlah fitur model: 22
Dimensi data preprocessed: (32407, 23)
Dimensi data cleaned: (32409, 12)


## 3. Inisialisasi SHAP TreeExplainer & Engine Ekstraksi Fitur

**Penjelasan Konsep:**
Agar prompt LLM memiliki landasan analitik (*grounded reasoning*), kita mengekstrak kontribusi SHAP lokal untuk setiap nasabah.
- **Top Risk Escalators**: Faktor-faktor dengan nilai SHAP positif tertinggi (+), yang mendorong naiknya probabilitas gagal bayar.
- **Top Mitigating Factors**: Faktor-faktor dengan nilai SHAP negatif terendah (-), yang meredam risiko dan memperkuat profil nasabah.

In [3]:
# Inisialisasi SHAP TreeExplainer
print("Menginisialisasi SHAP TreeExplainer...")
explainer = shap.TreeExplainer(model)

def extract_applicant_risk_signals(model, explainer, feature_row, feature_names, top_n=3):
    """
    Mengekstrak sinyal risiko kuantitatif (Probabilitas Default & Top Drivers SHAP)
    dari satu baris profil nasabah.
    
    Returns:
        dict: Berisi prob_default, risk_tier, top_risk_escalators, dan top_mitigating_factors
    """
    # Pastikan format input adalah 2D DataFrame
    if isinstance(feature_row, pd.Series):
        df_row = pd.DataFrame([feature_row])
    else:
        df_row = feature_row.copy()
        
    # 1. Hitung Probability of Default (PD)
    prob_default = float(model.predict_proba(df_row)[0, 1])
    
    # 2. Hitung SHAP Values
    shap_obj = explainer(df_row)
    shap_vals = shap_obj.values[0]
    
    # 3. Urutkan kontribusi fitur
    contribs = []
    for name, val, s_val in zip(feature_names, df_row.iloc[0].values, shap_vals):
        contribs.append({
            'feature': name,
            'value': float(val),
            'shap_value': float(s_val)
        })
    df_contrib = pd.DataFrame(contribs)
    
    risk_escalators = df_contrib[df_contrib['shap_value'] > 0].sort_values(by='shap_value', ascending=False).head(top_n).to_dict('records')
    mitigating_factors = df_contrib[df_contrib['shap_value'] < 0].sort_values(by='shap_value', ascending=True).head(top_n).to_dict('records')
    
    # Tentukan level risiko
    if prob_default < 0.20:
        risk_tier = 'LOW_RISK'
        prelim_decision = 'APPROVE'
    elif prob_default < 0.50:
        risk_tier = 'MEDIUM_RISK'
        prelim_decision = 'MANUAL_REVIEW'
    else:
        risk_tier = 'HIGH_RISK'
        prelim_decision = 'REJECT'
        
    return {
        'probability_of_default': round(prob_default, 4),
        'probability_of_default_pct': f"{prob_default * 100:.1f}%",
        'risk_tier': risk_tier,
        'prelim_decision': prelim_decision,
        'risk_escalators': risk_escalators,
        'mitigating_factors': mitigating_factors
    }

# Uji coba ekstraksi pada 1 sampel
sample_x = df_prep.drop(columns=['loan_status']).iloc[[0]]
sample_res = extract_applicant_risk_signals(model, explainer, sample_x, feature_names)
print("Hasil Uji Coba Ekstraksi Sinyal Risiko:")
print(json.dumps(sample_res, indent=2))

Menginisialisasi SHAP TreeExplainer...
Hasil Uji Coba Ekstraksi Sinyal Risiko:
{
  "probability_of_default": 0.0957,
  "probability_of_default_pct": "9.6%",
  "risk_tier": "LOW_RISK",
  "prelim_decision": "APPROVE",
  "risk_escalators": [
    {
      "feature": "person_income",
      "value": 9600.0,
      "shap_value": 3.146376132965088
    },
    {
      "feature": "disposable_income_est",
      "value": 8600.0,
      "shap_value": 1.2933505773544312
    },
    {
      "feature": "loan_intent_VENTURE",
      "value": 0.0,
      "shap_value": 0.13968351483345032
    }
  ],
  "mitigating_factors": [
    {
      "feature": "person_home_ownership_OWN",
      "value": 1.0,
      "shap_value": -2.333566665649414
    },
    {
      "feature": "loan_percent_income",
      "value": 0.1,
      "shap_value": -2.230581283569336
    },
    {
      "feature": "loan_grade_encoded",
      "value": 1.0,
      "shap_value": -0.5064094662666321
    }
  ]
}


## 4. Desain Template Prompt & Rule-Based Underwriting Logic

**Penjelasan Konsep:**
Dataset instruksi terdiri dari 3 kunci utama:
1. **`instruction`**: Memberi persona peran kepada LLM sebagai Senior Credit Underwriter dengan aturan format output JSON ketat.
2. **`input`**: Teks terformat yang merangkum data nasabah (usia, pendapatan, tujuan pinjaman, dll.) dan sinyal kalkulasi XGBoost + SHAP.
3. **`output`**: String JSON terstruktur yang berisi rekomendasi underwriting, evaluasi risiko, dan memo analisis profesional.

In [4]:
# Template System Instruction
SYSTEM_INSTRUCTION = (
    "Anda adalah Senior Credit Risk Underwriter AI di institusi perbankan. "
    "Tugas Anda adalah mengevaluasi aplikasi kredit pemohon berdasarkan data demografi, keuangan, "
    "hasil prediksi model XGBoost (Probability of Default), dan kontribusi faktor risiko matematis (SHAP Values).\n\n"
    "Hasilkan laporan analisis kredit (Credit Underwriting Memo) yang terstruktur strictly dalam format JSON valid."
)

def format_feature_narrative(feat_name, feat_val):
    """Menerjemahkan nama variabel teknis menjadi narasi finansial yang ramah dibaca."""
    descriptions = {
        'loan_percent_income': f"Rasio pinjaman terhadap pendapatan sebesar {feat_val:.1%}",
        'loan_int_rate': f"Suku bunga pinjaman sebesar {feat_val:.2f}%",
        'person_income': f"Pendapatan tahunan sebesar ${feat_val:,.0f}",
        'total_loan_cost': f"Total estimasi kewajiban pinjaman + bunga sebesar ${feat_val:,.0f}",
        'loan_amnt': f"Besaran plafon pinjaman sebesar ${feat_val:,.0f}",
        'person_emp_length': f"Masa kerja pemohon selama {feat_val:.1f} tahun",
        'person_age': f"Usia pemohon {int(feat_val)} tahun",
        'cb_person_default_on_file_encoded': "Memiliki catatan riwayat gagal bayar di biro kredit" if feat_val == 1 else "Bersih dari catatan gagal bayar di biro kredit",
        'loan_grade_encoded': f"Tingkat peringkat risiko kredit grade index {int(feat_val)}",
        'high_risk_flag': "Terdeteksi indikator risiko tinggi gabungan" if feat_val == 1 else "Tidak terdeteksi indikator risiko tinggi gabungan",
        'emp_to_age_ratio': f"Rasio kestabilan kerja terhadap usia {feat_val:.2f}",
        'cred_hist_to_age_ratio': f"Proporsi riwayat kredit terhadap usia {feat_val:.2f}",
        'disposable_income_est': f"Estimasi sisa pendapatan setelah pinjaman ${feat_val:,.0f}"
    }
    return descriptions.get(feat_name, f"{feat_name} bernilai {feat_val}")

def construct_input_prompt(raw_profile, risk_signals):
    """Menyusun bagian 'input' dari instruksi."""
    prompt_lines = [
        "### PROFIL PEMOHON PINJAMAN:",
        f"- Usia Pemohon: {int(raw_profile.get('person_age', 0))} tahun",
        f"- Pendapatan Tahunan: ${raw_profile.get('person_income', 0):,.0f}",
        f"- Status Kepemilikan Rumah: {raw_profile.get('person_home_ownership', 'N/A')}",
        f"- Lama Bekerja: {raw_profile.get('person_emp_length', 0):.1f} tahun",
        f"- Tujuan Pinjaman: {raw_profile.get('loan_intent', 'N/A')}",
        f"- Peringkat Risiko Kredit (Grade): {raw_profile.get('loan_grade', 'N/A')}",
        f"- Besaran Pinjaman yang Diajukan: ${raw_profile.get('loan_amnt', 0):,.0f}",
        f"- Suku Bunga Pinjaman: {raw_profile.get('loan_int_rate', 0):.2f}%",
        f"- Rasio Pinjaman / Pendapatan: {raw_profile.get('loan_percent_income', 0):.1%}",
        f"- Riwayat Gagal Bayar Sebelumnya: {raw_profile.get('cb_person_default_on_file', 'N')}",
        f"- Panjang Riwayat Kredit: {int(raw_profile.get('cb_person_cred_hist_length', 0))} tahun",
        "",
        "### KALKULASI RISIKO ML & ANALISIS SHAP:",
        f"- Prediksi Probability of Default (PD): {risk_signals['probability_of_default_pct']}",
        f"- Kategori Risiko Awal: {risk_signals['risk_tier']}",
        f"- Rekomendasi Awal: {risk_signals['prelim_decision']}",
        "- Faktor Pendorong Risiko Terbesar (+SHAP):"
    ]
    for esc in risk_signals['risk_escalators']:
        prompt_lines.append(f"  * {format_feature_narrative(esc['feature'], esc['value'])} (SHAP: +{esc['shap_value']:.3f})")
        
    prompt_lines.append("- Faktor Pereda Risiko Terbesar (-SHAP):")
    for mit in risk_signals['mitigating_factors']:
        prompt_lines.append(f"  * {format_feature_narrative(mit['feature'], mit['value'])} (SHAP: {mit['shap_value']:.3f})")
        
    return "\n".join(prompt_lines)

### 4.1 Rule-Based Target Memo Synthesis Engine

**Penjelasan Konsep:**
Fungsi ini mensintesis ground-truth JSON output (`output`) yang mematuhi standar memo underwriting profesional. Variasi narasi dibuat dinamis berdasarkan profil risiko pemohon.

In [5]:
def synthesize_ground_truth_memo(raw_profile, risk_signals):
    """
    Menghasilkan ground-truth output JSON memo underwriting.
    """
    pd_val = risk_signals['probability_of_default']
    decision = risk_signals['prelim_decision']
    tier = risk_signals['risk_tier']
    
    key_drivers = [format_feature_narrative(e['feature'], e['value']) for e in risk_signals['risk_escalators']]
    mitigating_factors = [format_feature_narrative(m['feature'], m['value']) for m in risk_signals['mitigating_factors']]
    
    # Sintesis narasi memo analisis
    if tier == 'HIGH_RISK':
        memo_text = (
            f"Aplikasi pinjaman ini dinilai berisiko tinggi dengan estimasi probabilitas gagal bayar {risk_signals['probability_of_default_pct']}. "
            f"Faktor risiko paling dominan adalah {key_drivers[0].lower() if key_drivers else 'beban utang yang tinggi'}. "
            f"Meskipun terdapat faktor pereda seperti {mitigating_factors[0].lower() if mitigating_factors else 'tidak ada faktor signifikan'}, "
            f"profil pemohon secara keseluruhan melampaui toleransi risiko standar perbankan. Disarankan penolakan pinjaman atau pengajuan agunan tambahan."
        )
        conditions = ["Pengajuan agunan bernilai likuid tinggi", "Penambahan penjamin kredit (guarantor)", "Pengurangan plafon pinjaman minimal 50%"]
    elif tier == 'MEDIUM_RISK':
        memo_text = (
            f"Aplikasi pinjaman berada dalam kategori risiko moderat (PD: {risk_signals['probability_of_default_pct']}). "
            f"Perhatian khusus diperlukan terhadap {key_drivers[0].lower() if key_drivers else 'aspek beban kewajiban'}. "
            f"Namun demikian, pemohon didukung oleh {mitigating_factors[0].lower() if mitigating_factors else 'stabilitas data'}. "
            f"Rekomendasi persetujuan bersyarat dengan penyesuaian plafon atau verifikasi penghasilan lanjutan oleh tim underwriting."
        )
        conditions = ["Verifikasi slip gaji dan mutasi rekening 3 bulan terakhir", "Penyesuaian jangka waktu tenor untuk menekan rasio angsuran bulanan"]
    else:
        memo_text = (
            f"Aplikasi pinjaman memiliki profil risiko yang sangat sehat dengan estimasi probabilitas gagal bayar rendah ({risk_signals['probability_of_default_pct']}). "
            f"Kekuatan utama pemohon didorong oleh {mitigating_factors[0].lower() if mitigating_factors else 'kapasitas finansial yang kuat'}. "
            f"Pengajuan direkomendasikan untuk disetujui (APPROVE) sesuai dengan ketentuan standar suku bunga dan plafon pinjaman."
        )
        conditions = ["Persetujuan instan sesuai plafon yang diajukan", "Penerapan suku bunga standar kelas risiko prima"]
        
    memo_json = {
        "recommendation": decision,
        "risk_tier": tier,
        "probability_of_default": risk_signals['probability_of_default_pct'],
        "key_risk_drivers": key_drivers,
        "mitigating_factors": mitigating_factors,
        "special_conditions": conditions,
        "underwriter_memo": memo_text
    }
    
    return json.dumps(memo_json, ensure_ascii=False, indent=2)

## 5. Sintesis & Sampling Dataset (Balanced 1,000+ Records)

**Penjelasan Konsep:**
Agar model LLM terlatih mengenali berbagai variasi kasus secara seimbang:
- Kita mengambil sampel berimbang dari kategori **High Risk / Default (Target = 1)** dan **Low-Medium Risk / Non-Default (Target = 0)**.
- Target ukuran dataset sintetis: **1.000 hingga 1.500 sampel** (ukuran ideal untuk fine-tuning QLoRA hemat VRAM tanpa overfitting).

In [6]:
# Tentukan jumlah sampel yang ingin disintesis
NUM_SAMPLES = 1200
samples_per_class = NUM_SAMPLES // 2

# Filter indeks berdasarkan kelas aktual
idx_default = df_prep[df_prep['loan_status'] == 1].index.tolist()
idx_non_default = df_prep[df_prep['loan_status'] == 0].index.tolist()

# Sampling acak yang seimbang
selected_default = random.sample(idx_default, min(samples_per_class, len(idx_default)))
selected_non_default = random.sample(idx_non_default, min(samples_per_class, len(idx_non_default)))
selected_indices = selected_default + selected_non_default
random.shuffle(selected_indices)

print(f"Total sampel yang dipilih untuk disintesis: {len(selected_indices):,} baris")
print(f"- Kelas Default (1)    : {len(selected_default):,} sampel")
print(f"- Kelas Non-Default (0): {len(selected_non_default):,} sampel")

Total sampel yang dipilih untuk disintesis: 1,200 baris
- Kelas Default (1)    : 600 sampel
- Kelas Non-Default (0): 600 sampel


In [7]:
# TODO: Loop generasi dataset instruksi
instruction_dataset = []

print("Memulai sintesis instruksi dataset...")
for idx in tqdm(selected_indices):
    # Ambil baris numerik untuk inferensi ML & SHAP
    feat_row = df_prep.drop(columns=['loan_status']).loc[[idx]]
    # Ambil profil mentah untuk teks narasi
    raw_profile = df_clean.loc[idx].to_dict()
    
    # Ekstrak sinyal risiko
    risk_signals = extract_applicant_risk_signals(model, explainer, feat_row, feature_names)
    
    # Susun input prompt & output memo
    input_text = construct_input_prompt(raw_profile, risk_signals)
    output_json_str = synthesize_ground_truth_memo(raw_profile, risk_signals)
    
    instruction_dataset.append({
        "instruction": SYSTEM_INSTRUCTION,
        "input": input_text,
        "output": output_json_str
    })

print(f"\n✓ Sintesis selesai! Berhasil membuat {len(instruction_dataset):,} record instruksi.")

Memulai sintesis instruksi dataset...


100%|██████████| 1200/1200 [00:48<00:00, 24.86it/s]


✓ Sintesis selesai! Berhasil membuat 1,200 record instruksi.


## 6. Validasi Kualitas Data & JSON Schema Verification

**Penjelasan Konsep:**
Sebelum data digunakan untuk melatih LLM di Google Colab (Unsloth), kita wajib memvalidasi:
1. Seluruh string output dapat di-parse sebagai JSON valid tanpa error sintaks.
2. Tidak ada nilai `None` atau teks kosong.
3. Distribusi panjang karakter teks berada dalam batas konteks (*token budget*).

In [8]:
# Validasi parsing JSON untuk seluruh sampel
valid_count = 0
invalid_count = 0
lengths = []

for i, item in enumerate(instruction_dataset):
    try:
        parsed = json.loads(item['output'])
        assert 'recommendation' in parsed
        assert 'probability_of_default' in parsed
        valid_count += 1
        lengths.append(len(item['instruction']) + len(item['input']) + len(item['output']))
    except Exception as e:
        invalid_count += 1

print(f"Valid JSON samples  : {valid_count:,} ({valid_count/len(instruction_dataset)*100:.1f}%)")
print(f"Invalid JSON samples: {invalid_count}")
print(f"Rata-rata panjang karakter prompt total: {np.mean(lengths):.0f} karakter (Min: {min(lengths)}, Max: {max(lengths)})")

# Tampilkan 1 sampel instruksi lengkap
print("\n=== CONTOH SAMPEL INSTRUKSI PERTAMA ===")
print("[INSTRUCTION]:", instruction_dataset[0]['instruction'][:100], "...")
print("\n[INPUT]:\n", instruction_dataset[0]['input'])
print("\n[OUTPUT]:\n", instruction_dataset[0]['output'])

Valid JSON samples  : 1,200 (100.0%)
Invalid JSON samples: 0
Rata-rata panjang karakter prompt total: 2348 karakter (Min: 2134, Max: 2496)

=== CONTOH SAMPEL INSTRUKSI PERTAMA ===
[INSTRUCTION]: Anda adalah Senior Credit Risk Underwriter AI di institusi perbankan. Tugas Anda adalah mengevaluasi ...

[INPUT]:
 ### PROFIL PEMOHON PINJAMAN:
- Usia Pemohon: 22 tahun
- Pendapatan Tahunan: $55,000
- Status Kepemilikan Rumah: RENT
- Lama Bekerja: 3.0 tahun
- Tujuan Pinjaman: PERSONAL
- Peringkat Risiko Kredit (Grade): D
- Besaran Pinjaman yang Diajukan: $15,000
- Suku Bunga Pinjaman: 14.61%
- Rasio Pinjaman / Pendapatan: 27.0%
- Riwayat Gagal Bayar Sebelumnya: Y
- Panjang Riwayat Kredit: 4 tahun

### KALKULASI RISIKO ML & ANALISIS SHAP:
- Prediksi Probability of Default (PD): 27.5%
- Kategori Risiko Awal: MEDIUM_RISK
- Rekomendasi Awal: MANUAL_REVIEW
- Faktor Pendorong Risiko Terbesar (+SHAP):
  * Tingkat peringkat risiko kredit grade index 3 (SHAP: +1.375)
  * person_home_ownership_RENT bern

## 7. Export ke Format JSONL (`data/credit_finetune_dataset.jsonl`)

**Penjelasan Konsep:**
Format **JSONL (JSON Lines)** menyimpan 1 objek JSON per baris teks. Format ini merupakan format standar industri untuk fine-tuning menggunakan pustaka **Unsloth**, **Hugging Face TRL (SFTTrainer)**, dan **OpenAI Fine-Tuning API**.

In [9]:
output_jsonl_path = '../data/credit_finetune_dataset.jsonl' if os.path.exists('../data') else ('data/credit_finetune_dataset.jsonl' if os.path.exists('data') else 'credit_finetune_dataset.jsonl')

# Tulis data ke file JSONL
with open(output_jsonl_path, 'w', encoding='utf-8') as f:
    for entry in instruction_dataset:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"✓ Dataset fine-tuning berhasil disimpan ke '{output_jsonl_path}'")
print(f"  Ukuran file: {os.path.getsize(output_jsonl_path) / 1024:.2f} KB | Total baris: {len(instruction_dataset):,}")

print("\n=========================================================================")
print("🎉 FASE 2 (DATASET SYNTHESIS) SELESAI! SIAP UNTUK UNSLOTH FINE-TUNING (04)")
print("=========================================================================")

✓ Dataset fine-tuning berhasil disimpan ke '../data/credit_finetune_dataset.jsonl'
  Ukuran file: 2906.77 KB | Total baris: 1,200

🎉 FASE 2 (DATASET SYNTHESIS) SELESAI! SIAP UNTUK UNSLOTH FINE-TUNING (04)
